# Working With Databases

Often data is stored in databases. Especially in companies. There are different types of databases, but the most common one is the relational database. In a relational database, data is stored in tables. Each table has a primary key, which is a unique identifier for each row. The tables are related to each other through foreign keys, which are references to the primary keys of other tables. This is how the relations are implemented. To access the data in a relational database, you can use SQL (Structured Query Language).

If you want to learn more about SQL you can check out this tutorial: [SQL Tutorial](https://www.w3schools.com/sql/default.asp)

There are a lot of libraries in Python to work with databases. It depends on what you want to archive with it which library you should use. If you are working with data in an application you might want to use an Object-Relational Mapping (ORM) library like SQLAlchemy. If you just want to run some queries on a database to get data for analysis you don't need the full power of an ORM. None the less, I will show you how to use SQLAlchemy to connect to a database and run some queries. SQLAlchemy has the ability to work with different types of databases, so you can use it to connect to a MySQL, PostgreSQL or SQLite database.

## Setup

Before you can start you maybe need to install some packages if not already done.

In [ ]:
%pip install -r ../requirements.txt

from contextlib import contextmanager
from pathlib import Path
from sqlalchemy import Engine, create_engine, text
from sqlalchemy.orm import sessionmaker, Session, declarative_base
from typing import Any, Generator

## Reading Data With SQLAlchemy

Because here will only want to read data and not manipulate it, I will show some code I extracted from another project of mine to make working with the database easier. You can find the code of a more complete example in this repository: [iBrotNano/python\_workshop\_project: This is a simple training project for a Python workshop.](https://github.com/iBrotNano/python_workshop_project).

In [ ]:
# You can copy this code and use it to set up a database engine for your project. 
# It encapsulates the SQLAlchemy engine, session, and base for database operations. 
# The DatabaseEngineFactory class is responsible for configuring the database engine with the necessary parameters.
class DatabaseEngine:
    """
    Encapsulates the SQLAlchemy engine, session, and base for database operations.
    """

    def __init__(self, engine: Engine, session: sessionmaker, base: Any):
        """
        Initializes the DatabaseEngine with the provided SQLAlchemy engine, session, and base.

        :param engine: The SQLAlchemy engine instance.
        :type engine: Engine
        :param session: The SQLAlchemy sessionmaker instance.
        :type session: sessionmaker
        :param base: The SQLAlchemy declarative base.
        :type base: Any
        """
        self._engine: Engine = engine
        self.session: sessionmaker = session
        self.Base: Any = base

    @property
    def engine(self) -> Engine:
        """
        Gets the SQLAlchemy engine instance.

        :return: The SQLAlchemy engine.
        :rtype: Engine
        """
        return self._engine

    @contextmanager
    def get_db(self) -> Generator[Session, None, None]:
        """
        Provides a database session for performing operations.
        This method is a generator that yields a session and ensures
        it is properly closed after use.

        By decorating this method with @contextmanager, it can be used in a
        with statement to automatically manage the session's lifecycle.

        :return: A generator yielding a SQLAlchemy session.
        :rtype: Generator[Session, None, None]
        """
        db = self.session()

        try:
            yield db
        finally:
            db.close()

class DatabaseEngineFactory:
    """
    Configures the database engine with the necessary parameters.
    This class is responsible for setting up the database engine using the configuration defined in the configuration module.
    """

    @staticmethod
    def create(sqlite_url: str, sqlite_auto_flush: bool) -> DatabaseEngine:
        """
        Configures and returns an instance of the DatabaseEngine.

        :return: An instance of the DatabaseEngine class.
        :rtype: DatabaseEngine
        """
        engine = create_engine(
            sqlite_url
        )

        session = sessionmaker(
            autoflush=sqlite_auto_flush,
            bind=engine,
        )

        base = declarative_base()
        return DatabaseEngine(engine, session, base)

# Builds the path to the SQLite database file.
db_path = Path.cwd().parent / "assets" / "db" / "nutrition_data.db"

# Now you can use it to initialize the database engine for your project.
# Here you can configure URLs to other database types as well.
database_engine = DatabaseEngineFactory().create(f"sqlite:///{db_path.as_posix()}", True)

# Now you can use 'with' to get a database session.
# It handles the opening and closing of the session for you, ensuring that resources are properly managed.
with database_engine.get_db() as session:
    result = session.execute(text("SELECT code, name, url FROM nutrition WHERE name LIKE :name"), {"name": "%Coca Cola%"}).fetchall()
    
    for row in result:
        print(f"Code: {row.code}, Name: {row.name}, URL: {row.url}")
